# 🎓 Lab Tutorial: Custom Schema GraphRAG with Neo4j (LlamaIndex + Ollama)

You now have separate notebooks for each storage/data model:

1. **[2.graphrag_custom_schema_tutorial.ipynb](file:///home/ashok/Documents/graph_rag/2.graphrag_custom_schema_tutorial.ipynb)** — Pure in-memory (`SimplePropertyGraphStore`) for both schema-guided extraction and manual `EntityNode`/`Relation` construction.
2. **[3.graphrag_custom_schema_qdrant.ipynb](file:///home/ashok/Documents/graph_rag/3.graphrag_custom_schema_qdrant.ipynb)** — Standalone Qdrant Docker vector store + persisted graph store notebook.
3. **[4.graphrag_custom_schema_neo4j.ipynb](file:///home/ashok/Documents/graph_rag/4.graphrag_custom_schema_neo4j.ipynb)** — Standalone Neo4j notebook using `Neo4jPropertyGraphStore`:
   - Connects to Neo4j (`bolt://localhost:7687`) with an optional `CLEAR_DATABASE = True` toggle.
   - **Approach 1:** `SchemaLLMPathExtractor` stored directly in Neo4j, Cypher inspection, and fast single-LLM-call querying with `VectorContextRetriever`.
   - **Approach 2:** Manual `EntityNode` & `Relation` upserts directly into Neo4j via `upsert_nodes()` / `upsert_relations()` and `PropertyGraphIndex.from_existing(...)`.

In [1]:
# Uncomment and run once if not already installed:
# %pip install llama-index-graph-stores-neo4j

#### Install neo4j first

This notebook mirrors `2.graphrag_custom_schema_tutorial.ipynb`, but replaces the in-memory `SimplePropertyGraphStore` with a persistent **Neo4j** graph database (`Neo4jPropertyGraphStore`).

> **Prerequisite — Make sure Neo4j is running (e.g., via Docker):**
> ```bash
> docker run -d --name neo4j -p 7474:7474 -p 7687:7687 \
>   -e NEO4J_AUTH=neo4j/password123 \
>   -e NEO4J_PLUGINS='["apoc"]' neo4j:latest
> ```
> You can visually browse your graph anytime at **http://localhost:7474**.      
> Also browse its logs as:
> ```bash
> docker logs neo4j
> ```

---

### How it works?
 When you run `docker run ... neo4j:latest`, Docker will **automatically download (`pull`)** the Neo4j image the first time, create the container, and start it in the background (`-d`).

*(Optional)* If you want your Neo4j database files to persist on your host folder even if you delete the container later, add a volume mount (`-v`):

```bash
docker run -d --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -v $(pwd)/neo4j_data:/data \
  -e NEO4J_AUTH=neo4j/password123 \
  -e NEO4J_PLUGINS='["apoc"]' \
  neo4j:latest
```

> **Note:** After the first `docker run`, give Neo4j ~20 seconds to download APOC and finish booting before running the Jupyter notebook (you can verify with `docker logs neo4j` until you see `Started.`).

---



Here No password is needed (still includes the APOC plugin required by LlamaIndex):

```bash
docker run -d --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -v $(pwd)/neo4j_data:/data \
  -e NEO4J_AUTH=none \
  -e NEO4J_PLUGINS='["apoc"]' \
  neo4j:latest
```

#### 2. How to Stop, Start, and Restart neo4j
Once created, you do **not** run `docker run` again — use the container name `neo4j`:

| Action | Command |
| :--- | :--- |
| **Stop Neo4j** | `docker stop neo4j` |
| **Start it again** (after stopping or reboot) | `docker start neo4j` |
| **Restart it** | `docker restart neo4j` |
| **Check if it is running** | `docker ps` |
| **Check startup logs** | `docker logs neo4j` |
| **Delete the container** (only if recreating) | `docker rm -f neo4j` |

Once running, you can also open the visual Neo4j Browser at **http://localhost:7474** (username: `neo4j`, password: `password123`).

## 1. Setup & Environment Configuration

In [1]:
%%time

# CRITICAL FOR JUPYTER NOTEBOOKS:
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Paths
DATA_DIR = Path.cwd() / "data"

# Configure Local Ollama LLM (temperature=0.0 for deterministic schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# Load the 4 documents from data/
documents = SimpleDirectoryReader(input_dir=str(DATA_DIR)).load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

Loaded 4 documents from /home/ashok/Documents/14.graph_rag/data
CPU times: user 4.28 s, sys: 71.6 ms, total: 4.35 s
Wall time: 953 ms


## 2. Connect to Neo4j (`Neo4jPropertyGraphStore`)

In [2]:
%%time

from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

NEO4J_URL = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"   # Match your Neo4j password
CLEAR_DATABASE = True            # Set False to keep existing Neo4j data

neo4j_store = Neo4jPropertyGraphStore(
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    url=NEO4J_URL,
)

if CLEAR_DATABASE:
    neo4j_store.structured_query("MATCH (n) DETACH DELETE n")
    print("Cleared existing Neo4j database for a clean run.")

print(f"Connected to Neo4j at {NEO4J_URL}")

Cleared existing Neo4j database for a clean run.
Connected to Neo4j at bolt://localhost:7687
CPU times: user 92.3 ms, sys: 14.9 ms, total: 107 ms
Wall time: 465 ms


---
## 3. Approach 1: Schema-Guided Extraction (`SchemaLLMPathExtractor`) into Neo4j

Pass `property_graph_store=neo4j_store` to `PropertyGraphIndex.from_documents(...)` so all extracted nodes, relationships, and embeddings are saved directly in Neo4j.

In [3]:
%%time

from typing import Literal
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor

# 1. Define allowed Node Types (Entities)
entities = Literal[
    "PERSON",
    "COMPANY",
    "INITIATIVE",
    "TECHNOLOGY",
    "LOCATION",
    "REGULATOR",
]

# 2. Define allowed Relationship Types (Edges)
relations = Literal[
    "CO_FOUNDED",
    "WORKS_AT",
    "ADVISES",
    "SERVES_ON_BOARD_OF",
    "COLLABORATED_WITH",
    "INVESTED_IN",
    "FUNDED",
    "JOINT_VENTURE_WITH",
    "LICENSES_TO",
    "DEVELOPS",
    "TESTBED_FOR",
    "HEADQUARTERED_IN",
    "INVESTIGATES",
]

# 3. Define valid (SourceEntityType, Relation, TargetEntityType) rules
validation_schema = [
    ("PERSON", "CO_FOUNDED", "COMPANY"),
    ("PERSON", "WORKS_AT", "COMPANY"),
    ("PERSON", "ADVISES", "COMPANY"),
    ("PERSON", "SERVES_ON_BOARD_OF", "COMPANY"),
    ("PERSON", "COLLABORATED_WITH", "PERSON"),
    ("PERSON", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "INVESTED_IN", "COMPANY"),
    ("COMPANY", "FUNDED", "INITIATIVE"),
    ("COMPANY", "JOINT_VENTURE_WITH", "COMPANY"),
    ("COMPANY", "LICENSES_TO", "COMPANY"),
    ("COMPANY", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "TESTBED_FOR", "INITIATIVE"),
    ("COMPANY", "HEADQUARTERED_IN", "LOCATION"),
    ("REGULATOR", "INVESTIGATES", "PERSON"),
    ("REGULATOR", "INVESTIGATES", "COMPANY"),
]

# 4. Create the Schema-Constrained Extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True,
    max_triplets_per_chunk=10,
    num_workers=1,
)

print("Constructing Schema-Guided Property Graph in Neo4j...")
index_neo4j = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    property_graph_store=neo4j_store,
    use_async=False,
    show_progress=True,
)
print("Graph stored in Neo4j! Open http://localhost:7474 to inspect it visually.")

Constructing Schema-Guided Property Graph in Neo4j...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-26 15:02:29,292 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-26 15:03:09,400 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:03:44,140 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:04:23,632 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:05:00,196 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-11-26 15:05:02,396 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/48 [00:00<?, ?it/s]

2026-11-26 15:05:04,407 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:05:06,407 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:05:08,404 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:05:10,366 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:05:11,924 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Graph stored in Neo4j! Open http://localhost:7474 to inspect it visually.
CPU times: user 208 ms, sys: 16.3 ms, total: 224 ms
Wall time: 2min 43s


### 3.1 Inspect Stored Triplets in Neo4j via Cypher

In [4]:
%%time

rows = neo4j_store.structured_query(
    "MATCH (a)-[r]->(b) WHERE NOT a:Chunk AND NOT b:Chunk "
    "RETURN a.name AS source, labels(a) AS src_labels, type(r) AS relation, "
    "b.name AS target, labels(b) AS tgt_labels LIMIT 30"
)

print(f"Extracted entity relationships in Neo4j: {len(rows)}\n")
for row in rows:
    print(f"({row['source']} {row['src_labels']}) --[{row['relation']}]--> ({row['target']} {row['tgt_labels']})")

Extracted entity relationships in Neo4j: 17

(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[INVESTED_IN]--> (Synapse Dynamics ['__Node__', '__Entity__', 'COMPANY'])
(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[LICENSES_TO]--> (Helios Capital ['__Node__', '__Entity__', 'COMPANY'])
(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[LICENSES_TO]--> (Synapse Dynamics ['__Node__', '__Entity__', 'COMPANY'])
(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[JOINT_VENTURE_WITH]--> (Aether Labs ['__Node__', '__Entity__', 'COMPANY'])
(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[JOINT_VENTURE_WITH]--> (Helios Capital ['__Node__', '__Entity__', 'COMPANY'])
(Aurora Technologies Inc. ['__Node__', '__Entity__', 'COMPANY']) --[HEADQUARTERED_IN]--> (Seattle, Washington ['__Node__', '__Entity__', 'LOCATION'])
(Helios Capital ['__Node__', '__Entity__', 'COMPANY']) --[INVESTED_IN]--> (Aether Labs ['__Node

### 3.2 Query Neo4j via LlamaIndex (Fast Single-LLM-Call Vector + Graph Retrieval)

In [5]:
%%time

from llama_index.core.indices.property_graph import VectorContextRetriever

query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

# Use Neo4j's native vector search + graph expansion directly (1 Ollama /api/chat call)
neo4j_retriever = VectorContextRetriever(
    index_neo4j.property_graph_store,
    embed_model=embed_model,
    similarity_top_k=2,
    path_depth=1,
    include_text=True,
)

neo4j_query_engine = index_neo4j.as_query_engine(
    sub_retrievers=[neo4j_retriever],
    use_async=False,
)
response_neo4j = neo4j_query_engine.query(query_text)

print("=== 💡 NEO4J GRAPHRAG RESPONSE ===")
print(response_neo4j.response)

2026-11-26 15:05:12,406 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:05:35,853 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 NEO4J GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics and Project Nexus through Helios Capital in the following ways:

1. **Synapse Dynamics Connection**: Dr. Elena Vance, along with Dr. Kenji Sato, co-authored foundational papers on quantum robotics at Stanford University in 2016. Vance brought Aurora Technologies' proprietary CryoQubit compression research to Synapse Dynamics, where Sato integrated it into the humanoid bodies of their robots. Additionally, Helios Capital invested in Synapse Dynamics, making it part of the broader network of companies involved in Project Nexus.

2. **Project Nexus Connection**: Helios Capital, which invested in Aurora Technologies, played a crucial role in stitching together the entities involved in Project Nexus. Julian Mercer, who used his positions at Helios Capital to sanction undisclosed IP transfers, facilitated the integration of Aurora Technologies' CryoQubit technology with Synapse Dynamics' humanoid bodies. This 

---
## 4. Approach 2: Manually Upserting Exact Nodes & Relationships into Neo4j

If you want 100% deterministic control over every node and edge in Neo4j (without LLM extraction), upsert `EntityNode` and `Relation` objects directly into `neo4j_store` and wrap it with `PropertyGraphIndex.from_existing(...)`.

In [ ]:
from llama_index.core.graph_stores import EntityNode, Relation

# Optional: clear Neo4j first if you want ONLY the manual graph
# neo4j_store.structured_query("MATCH (n) DETACH DELETE n")

elena = EntityNode(name="Dr. Elena Vance", label="PERSON", properties={"role": "Co-Founder & Chief Scientific Advisor"})
marcus = EntityNode(name="Marcus Thorne", label="PERSON", properties={"role": "Co-Founder & CEO"})
kenji = EntityNode(name="Dr. Kenji Sato", label="PERSON", properties={"role": "Founder & CEO"})
julian = EntityNode(name="Julian Mercer", label="PERSON", properties={"role": "Principal Investment Director"})
aurora = EntityNode(name="Aurora Technologies", label="COMPANY", properties={"headquarters": "Seattle, WA", "founded": 2021})
helios = EntityNode(name="Helios Capital", label="COMPANY", properties={"headquarters": "New York City"})
aether = EntityNode(name="Aether Labs", label="COMPANY", properties={"type": "Joint Venture"})
synapse = EntityNode(name="Synapse Dynamics", label="COMPANY", properties={"headquarters": "Boston", "founded": 2019})
nexus = EntityNode(name="Project Nexus", label="INITIATIVE", properties={"status": "Stealth consortium"})

nodes = [elena, marcus, kenji, julian, aurora, helios, aether, synapse, nexus]

relations = [
    Relation(label="CO_FOUNDED", source_id=elena.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="CO_FOUNDED", source_id=marcus.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="ADVISES", source_id=elena.id, target_id=synapse.id, properties={"since": "March 2024"}),
    Relation(label="COLLABORATED_WITH", source_id=elena.id, target_id=kenji.id, properties={"year": 2016}),
    Relation(label="CO_FOUNDED", source_id=kenji.id, target_id=synapse.id, properties={"year": 2019}),
    Relation(label="WORKS_AT", source_id=julian.id, target_id=helios.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aurora.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aether.id),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aurora.id, properties={"amount": "$30M"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=synapse.id, properties={"amount": "$60M"}),
    Relation(label="JOINT_VENTURE_WITH", source_id=aurora.id, target_id=aether.id),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aether.id, properties={"amount": "$45M"}),
    Relation(label="LICENSES_TO", source_id=aurora.id, target_id=synapse.id),
    Relation(label="FUNDED", source_id=helios.id, target_id=nexus.id, properties={"mode": "Covert"}),
    Relation(label="TESTBED_FOR", source_id=synapse.id, target_id=nexus.id),
]

neo4j_store.upsert_nodes(nodes)
neo4j_store.upsert_relations(relations)

index_neo4j_manual = PropertyGraphIndex.from_existing(
    property_graph_store=neo4j_store,
    llm=llm,
    embed_model=embed_model,
    use_async=False,
)
print(f"Upserted {len(nodes)} nodes and {len(relations)} relationships into Neo4j!")

In [ ]:
# Query the Neo4j graph
manual_neo4j_engine = index_neo4j_manual.as_query_engine(include_text=False, use_async=False)
response_manual = manual_neo4j_engine.query(
    "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"
)

print("=== 💡 MANUAL NEO4J GRAPHRAG RESPONSE ===")
print(response_manual.response)